# Week 3 — live coding (completed)

**Social and Cultural Analytics (7AAVBCS5)**

## Setup

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a file or folder in the module's data folder."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        local = candidate / "data" / name
        if local.exists():
            return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    target = downloaded / name
    if not target.exists():
        raise FileNotFoundError(f"{name!r} is not in the module data.")
    return target


print("we are running in:", Path.cwd())

## 1. Creating a data frame

In [ ]:
places = pd.DataFrame({
    "name": ["Barbican Centre", "BFI Southbank", "Tate Modern", "Roundhouse"],
    "borough": ["City of London", "Lambeth", "Southwark", "Camden"],
    "venue_type": ["Arts centre", "Cinema", "Museum/gallery", "Music venue"],
    "visited": [True, True, True, False],
})
places

In [ ]:
print("shape:", places.shape)
print("columns:", list(places.columns))
places.info()

## 2. Files and loading borough context

We load the borough context CSV with `pd.read_csv()` through `data_path()`, then inspect it. One row is one London borough. These columns are meaningful to summarise: counts, population, area, income, transport accessibility and rates.

In [ ]:
borough_context = pd.read_csv(
    data_path("london_cultural_infrastructure_2023_borough_context.csv")
)
print("shape:", borough_context.shape)
print("columns:", list(borough_context.columns))
borough_context.head()

In [ ]:
borough_context.info()

In [ ]:
borough_context[[
    "venues",
    "population_2015",
    "venues_per_10k_residents",
    "venues_per_sq_km",
    "median_household_income_2012_13",
    "ptal_score_2014",
]].describe().round(1)

This summary tells us something. Cultural infrastructure is uneven: the standard deviation of
venue counts is larger than the mean, and the maximum rate per resident is an outlier.

In [ ]:
borough_context[[
    "borough_name",
    "venues",
    "population_2015",
    "venues_per_10k_residents",
    "venues_per_sq_km",
]].sort_values("venues_per_10k_residents", ascending=False).head(10)

City of London is a useful warning: a small residential population makes the per-resident rate
huge. That is not a bug. It is exactly why denominators matter.

## 3. Selecting and filtering borough data

In [ ]:
borough_context["venues_per_10k_residents"].head()

In [ ]:
borough_context[[
    "borough_name",
    "venues",
    "venues_per_10k_residents",
    "median_household_income_2012_13",
]].head()

In [ ]:
high_access = borough_context[
    borough_context["venues_per_10k_residents"] > 10
]
high_access[["borough_name", "venues_per_10k_residents"]]

In [ ]:
high_access_lower_income = borough_context[
    (borough_context["venues_per_10k_residents"] > 10) &
    (borough_context["median_household_income_2012_13"] < 50000)
]
high_access_lower_income[[
    "borough_name",
    "venues_per_10k_residents",
    "median_household_income_2012_13",
]]

## 4. `.loc` and `.iloc` — where it matters

In [ ]:
print("loc :", borough_context.loc[0, "borough_name"])
print("iloc:", borough_context.iloc[0, 0])

In [ ]:
lower_income = borough_context[
    borough_context["median_household_income_2012_13"] < 50000
]
print("first five labels:", list(lower_income.index[:5]))
print("iloc[0] borough:", lower_income.iloc[0]["borough_name"])

The labels came with the rows. `.iloc[0]` means first row in this filtered table; `.loc[0]`
means the row labelled 0.

In [ ]:
lower_income_clean = lower_income.reset_index(drop=True)
print("after reset_index:", list(lower_income_clean.index[:5]))

## 5. Raw and aggregated files, then saving output

In [ ]:
venues = pd.read_csv(data_path("london_cultural_venues_2023.csv"))
print("borough-level context file:", borough_context.shape)
print("venue-level file:", venues.shape)
venues[["name", "venue_type", "borough_name", "ward_name"]].head()

The borough file is aggregated and contextual. The venue file is closer to raw data: one row per
venue.

In [ ]:
high_access.to_csv("high_access_boroughs.csv", index=False)
check = pd.read_csv("high_access_boroughs.csv")
print(check.shape)
print(list(check.columns))

No `Unnamed: 0`, because we passed `index=False`.

## 6. Debugging with AI

In [ ]:
# Wrong capitalisation. Uncomment to see the KeyError.
# borough_context["Borough_name"]

In [ ]:
# `and` instead of `&`. Uncomment to see the ValueError.
# borough_context[
#     (borough_context["venues_per_10k_residents"] > 10) and
#     (borough_context["median_household_income_2012_13"] < 50000)
# ]

In [ ]:
# The version that works.
borough_context[
    (borough_context["venues_per_10k_residents"] > 10) &
    (borough_context["median_household_income_2012_13"] < 50000)
].shape

## What this dataset can and cannot tell us

The borough context file is good for comparing boroughs. It cannot tell us which specific venues
are in Camden or Hackney. For that, we need the venue-level file.

End of live coding.